# 🧬 Example 1: Clinical Trial GxP Synthesis, 3x Multi-Judge Consensus & FDA 21 CFR Part 11 Audit Ledger

This Jupyter Notebook demonstrates the clinical governance pipeline inside **`quantum-biomed-research-engine`**:
1. **HIPAA Safe Harbor & GDPR PHI Scrubbing (`PHIScrubber`)**
2. **Clinical Trial Protocol Synthesis (`ClinicalTrialSynthesizer`)** with `[CIT-P1]` / `[CIT-P2]` anchors
3. **3x Concurrent GxP Multi-Judge Consensus (`CalibratedMultiJudgeConsensus`)**
4. **FDA 21 CFR Part 11 Cryptographic Hash-Chain Ledger (`FDAPart11Ledger`)** + HITL Electronic Signature (`§11.50`)

In [ ]:
import os
import sys
import json

sys.path.insert(0, os.path.abspath('..'))

from src.privacy.phi_scrubber import PHIScrubber
from src.agents.clinical_synthesizer import ClinicalTrialSynthesizer
from src.evaluators.multi_judge_consensus import CalibratedMultiJudgeConsensus
from src.compliance.fda_part11_ledger import FDAPart11Ledger
from src.schemas import HITLApprovalRequest


## 1. Deterministic HIPAA / GDPR PHI De-Identification
Before any clinical protocol excerpt is sent to an LLM gateway, `PHIScrubber` redacts Patient Names, DOBs, MRNs, SSNs/CNPs, Emails, and Phones.

In [ ]:
raw_protocol = (
    "Patient Name: Johnathan Miller, DOB: 1968-04-19, MRN: ONC-884920, SSN: 412-88-9021, "
    "Email: j.miller68@clinical.org. Cohort A enrolled subjects with locally advanced KRAS-G12C "
    "mutated NSCLC receiving oral covalent inhibitor 960 mg QD. Safety monitoring recorded "
    "Grade 2 nausea, mild diarrhea, fatigue, and transient elevated ALT/AST transaminases."
)

scrubber = PHIScrubber()
scrubbed_payload = scrubber.scrub(raw_protocol)
print(f"Total PHI Redactions: {scrubbed_payload.total_redactions}")
print(f"Redaction Breakdown:  {scrubbed_payload.redaction_breakdown}")
print(f"Scrubbed Text:\n{scrubbed_payload.scrubbed_text}")

## 2. Clinical Protocol Synthesis & 3x Concurrent GxP Multi-Judge Consensus
We synthesize the scrubbed protocol and evaluate it concurrently across `ClinicalGroundednessJudge`, `PharmacovigilanceSafetyJudge`, and `GxPComplianceJudge`.

In [ ]:
async def run_clinical_pipeline():
    synthesizer = ClinicalTrialSynthesizer()
    consensus_engine = CalibratedMultiJudgeConsensus()
    synthesis = await synthesizer.synthesize(
        clinical_study_id="STUDY-ONCO-KRAS-2026",
        target_biomarker="KRAS-G12C",
        scrubbed_text=scrubbed_payload.scrubbed_text,
    )
    consensus = await consensus_engine.evaluate(scrubbed_payload.scrubbed_text, synthesis)
    return synthesis, consensus

synthesis, consensus = await run_clinical_pipeline()
print("=== EXECUTIVE CLINICAL SYNTHESIS ===")
print(synthesis.executive_clinical_synthesis)
print(f"\nConsensus Score: {consensus.consensus_score} | Status: {consensus.governance_status}")

## 3. FDA 21 CFR Part 11 Cryptographic Hash-Chain Ledger & HITL Electronic Signature
We append the record to the immutable SQLite ledger, sign it with a Medical Reviewer Electronic Signature (`21 CFR §11.50`), and verify the SHA-256 chain.

In [ ]:
ledger = FDAPart11Ledger(db_path="/tmp/notebook_fda_part11.db")
rec = ledger.append_record(
    clinical_study_id="STUDY-ONCO-KRAS-2026",
    record_type="CLINICAL_SYNTHESIS",
    target_biomarker_or_receptor="KRAS-G12C",
    scrubbed_prompt_hash=scrubbed_payload.scrubbed_sha256,
    synthesis_output=synthesis.executive_clinical_synthesis,
    consensus_score=consensus.consensus_score,
    governance_status=consensus.governance_status,
)
signed_rec = ledger.sign_record_hitl(
    HITLApprovalRequest(
        record_id=rec.record_id,
        medical_reviewer_id="MD-ROCHE-4491",
        electronic_signature_meaning="Medical Monitor Safety Review & Release Approval (21 CFR 11.50)",
        approved=True,
        reviewer_notes="Verified biomarker stratification and hepatic monitoring schedule.",
    )
)
integrity = ledger.verify_ledger_integrity()
analytics = ledger.get_governance_analytics()
print(f"Signed Record ID: {signed_rec.record_id} | Linked Prev Hash: {signed_rec.previous_hash[:18]}...")
print(f"SHA-256 Chain Verified: {integrity.verified} ({integrity.total_records_checked} blocks)")
print("Overall & Grouped Judge Analytics:", json.dumps(analytics, indent=2))